In [29]:
import pandas as pd
import seaborn as sns

In [30]:
# Reading dataset and setting it up to display all columns
df_url = 'all_data_M_2023.xlsx'
df = pd.read_excel(ds_url)
pd.set_option('display.max_columns', None)

## Choosing the occuptations for analysis

https://www.bls.gov/oes/2023/may/oes_stru.htm#15-0000

Using the OEWS (Occupational Employment and Wage Statistics) I selected the occupations that were in fields involving AI, Data Science and Software Development. Which are:

| Code | Occupation |
|---|---|
| 15-2051 | Data Scientists |
| 15-1221 | Computer and Information Research Scientists |
| 15-2041 | Statisticians |
| 15-2031 | Operations Research Analysts |
| 15-1211 | Computer Systems Analysts |
| 15-1243 | Database Architects |
| 15-1242 | Database Administrators |
| 15-1252 | Software Developers |
| 15-1251 | Computer Programmers |
| 15-1253 | Software Quality Assurance Analysts and Testers |
| 15-1254 | Web Developers |
| 15-1299 | Computer Occupations, All Other |

### Occupations I left out

I excluded occupations in the same group that are less closely linked to AI, data science or software development, for example Computer User Support Specialists and Computer Network Support Specialists (help-desk and support work), Network Architects and Network and Computer Systems Administrators (IT infrastructure), Actuaries and Mathematicians (specialised maths roles outside the data/software focus), and Information Security Analysts and Web and Digital Interface Designers (related, but borderline). Keeping the set focused makes the comparisons more meaningful.






In [31]:
# Occupations codes
codes = ["15-2051",
         "15-1221",
         "15-2041",
         "15-2031",
         "15-1211",
         "15-1243",
         "15-1242",
         "15-1252",
         "15-1251",
         "15-1253",
         "15-1254",
         "15-1299"]

# Sub tech dataframe from the original dataframe, containing only AI, Data Science and Software Development occupations
tech_df = df[df["OCC_CODE"].isin(codes)].copy()


### Cross-industry rows only

Each row in the dataset is a summary for one area, one industry and one occupation. This means the same occupation shows up many times: once for all industries combined (`cross-industry`), and again for each separate industry (for example, Software Developers in banking or in healthcare).

The cross-industry row already includes all the workers in the industry rows. If I kept both, the same people would be counted twice.

I kept only the `cross-industry` rows, so each occupation has one row per area. This also fits what I want to find out: how many people work in these fields overall, and not in which industry.

**Limitation:** by keeping only cross-industry rows, I can't see which industries employ these workers using the sub dataframe.

In [33]:
# I_GROUP gives the industry level, so it picks all-industry rows cleanly;
tech_df = tech_df[tech_df["I_GROUP"] == "cross-industry"]

### Geographic level

The dataset covers several levels of geography in the same table: the whole U.S., individual states, metropolitan areas and non-metropolitan (rural) areas. These levels overlap. The U.S. figure already includes every state, and each state already includes its metro and non-metro areas.

If I added rows from different levels together, the same workers would be counted more than once. To avoid this, I split the data into separate tables using `AREA_TYPE`, and each chart uses only one of them:

| Table | `AREA_TYPE` | What the area code means |
|---|---|---|
| `national` | 1 | One occupation in the whole U.S. |
| `states` | 2 | One occupation in one state |
| `territories` | 3 | One occupatation is one territory |
| `metros` | 4 | One occupation in one metro area |
| `nonmetro` | 6 | One occupation in one non-metro area |



Source for the `AREA_TYPE` codes::
https://www.rdocumentation.org/packages/oews2021/versions/1.0.0/topics/oews2021

In [34]:
# Checking all area types in sub dataframe tech_df
tech_df.groupby("AREA_TYPE")["AREA_TITLE"].unique().apply(lambda x: list(x[:5]))

AREA_TYPE
1                                               [U.S.]
2     [Alabama, Alaska, Arizona, Arkansas, California]
3                        [Puerto Rico, Virgin Islands]
4    [Abilene, TX, Aguadilla-Isabela, PR, Akron, OH...
6    [Northeast Alabama nonmetropolitan area, South...
Name: AREA_TITLE, dtype: object

In [ ]:
# Create one subset per area type to use in the visualisations
national    = tech_df[tech_df["AREA_TYPE"] == 1].copy()
states      = tech_df[tech_df["AREA_TYPE"] == 2].copy()
territories = tech_df[tech_df["AREA_TYPE"] == 3].copy()
metros      = tech_df[tech_df["AREA_TYPE"] == 4].copy()
nonmetro    = tech_df[tech_df["AREA_TYPE"] == 6].copy() 
